In [1]:
# !unzip -q vehicles.zip -d data

In [1]:
import os
from ultralytics import YOLO
import shutil
import torch

In [2]:
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        DEVICE = ",".join(str(i) for i in range(num_gpus))   # e.g. "0,1,2,3"
    else:
        DEVICE = "0"
    print(f"CUDA available — {num_gpus} GPU(s) detected. Using device: {DEVICE}")
    for i in range(num_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    DEVICE = "cpu"
    print("No CUDA GPU found — falling back to CPU")

CUDA available — 1 GPU(s) detected. Using device: 0
  GPU 0: NVIDIA A100-SXM4-80GB


In [3]:
def train_vehicle_detector(yaml_file_path, base_model, total_epochs, image_resolution, DEVICE):
    model = YOLO(base_model)
    model.train(
        data=yaml_file_path,
        epochs=total_epochs,
        imgsz=image_resolution,
        batch = -1,
        workers=12,
        cache = True,
        device = DEVICE,
        project='outputs',
        name='yolo_training_run',
        exist_ok=True # Overwrites the directory if you re-run it
    )
    optimal_weights_path = os.path.join('outputs', 'yolo_training_run', 'weights', 'best.pt')
    print(f"Optimal weights: {optimal_weights_path}\n")
    return optimal_weights_path

def evaluate_held_out_test_set(weights_file_path, yaml_file_path):
    model = YOLO(weights_file_path)
    metrics = model.val(data=yaml_file_path, split='test')
    map_50 = metrics.box.map50
    map_50_95 = metrics.box.map
    print(f"mAP@0.5IoU:        {map_50:.4f}")
    print(f"mAP@[0.5:0.95]IoU: {map_50_95:.4f}\n")

def execute_visual_inference(weights_file_path, input_directory, output_project_dir, output_run_name):
    model = YOLO(weights_file_path)
    model.predict(
        source=input_directory,
        conf=0.25, # Minimum confidence threshold to render a box
        save=True,
        project=output_project_dir,
        name=output_run_name,
        exist_ok=True
    )

In [4]:
BASE_PATH = 'data/'
PATH_TO_YAML = os.path.join(BASE_PATH,'vehicles', 'data.yaml')
PATH_TO_TEST_IMAGES = os.path.join(BASE_PATH, 'vehicles', 'test', 'images')
PATH_TO_CAMPUS_IMAGES = os.path.join(BASE_PATH, 'campus_images')

In [5]:
# Hyperparameters
ARCHITECTURE = 'yolov8n.pt'
EPOCHS = 15
RESOLUTION = 640

In [6]:
final_weights_path = train_vehicle_detector(
        yaml_file_path=PATH_TO_YAML,
        base_model=ARCHITECTURE,
        total_epochs=EPOCHS,
        image_resolution=RESOLUTION,
        DEVICE=DEVICE
    )

Ultralytics 8.4.35 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/vehicles/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_training_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100

In [7]:
weight_path = 'runs/detect'
final_weights_path = os.path.join(weight_path, final_weights_path)

evaluate_held_out_test_set(
        weights_file_path=final_weights_path,
        yaml_file_path=PATH_TO_YAML
    )

Ultralytics 8.4.35 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1350.4±890.9 MB/s, size: 478.4 KB)
val: Scanning /teamspace/studios/this_studio/data/vehicles/test/labels.cache... 3716 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3716/3716 1.3Git/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 27, len(boxes) = 8102. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 233/233 8.0it/s 29.0s<0.1s
                   all       3716       8102      0.856      0.822      0.892      0.699
                   bus        975       1583      0.855      0.786       0.88      

In [8]:
execute_visual_inference(
        weights_file_path=final_weights_path,
        input_directory=PATH_TO_TEST_IMAGES,
        output_project_dir='assignment_outputs',
        output_run_name='test_set_visuals'
    )


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/3716 /teamspace/studios/this_studio/data/vehicles/test/images/-1-640x427_jpg.rf.724967f3d2e25fcaeb0b382faacf4496.jpg: 448x640 1 car, 92.2ms
image 2/3716 /teamspace/studios/this_studio/data/vehicles/test/images/-22_jpg.rf.68ba2882c130550b315a9e6bec045445.jpg: 480x640 1 truck, 72.7ms
image 3/3716 /teamspace/studios/this_studio/data/vehicles/test/images/-24_jpg.rf.eeb03cd8a330ac3ed83b26017fd826ce.jpg: 640x640 1 truck, 9.0ms
image 4/3716 /teamspace/s

In [9]:
!zip -r runs_backup.zip runs/

  adding: runs/ (stored 0%)
  adding: runs/detect/ (stored 0%)
  adding: runs/detect/val/ (stored 0%)
  adding: runs/detect/val/val_batch0_pred.jpg (deflated 11%)
  adding: runs/detect/val/confusion_matrix_normalized.png (deflated 20%)
  adding: runs/detect/val/val_batch1_labels.jpg (deflated 11%)
  adding: runs/detect/val/BoxPR_curve.png (deflated 8%)
  adding: runs/detect/val/BoxR_curve.png (deflated 8%)
  adding: runs/detect/val/val_batch0_labels.jpg (deflated 12%)
  adding: runs/detect/val/val_batch2_pred.jpg (deflated 13%)
  adding: runs/detect/val/BoxF1_curve.png (deflated 8%)
  adding: runs/detect/val/val_batch2_labels.jpg (deflated 13%)
  adding: runs/detect/val/val_batch1_pred.jpg (deflated 10%)
  adding: runs/detect/val/BoxP_curve.png (deflated 7%)
  adding: runs/detect/val/confusion_matrix.png (deflated 21%)
  adding: runs/detect/assignment_outputs/ (stored 0%)
  adding: runs/detect/assignment_outputs/test_set_visuals/ (stored 0%)
  adding: runs/detect/assignment_outputs/tes